# 01 - Sequences and Mutations

This notebook introduces the FUS low-complexity domain (LCD) sequences and demonstrates the mutation utilities.

## Contents
1. Load the canonical FUS LCD sequence (residues 1-214)
2. Explore sequence properties
3. Create and validate mutations
4. Build the variant registry
5. Save sequences for downstream analysis

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Import sequence utilities
from src.sequences import (
    FUS_LCD_SEQUENCE,
    SequenceRecord,
    VARIANTS,
    get_variant,
    list_variants,
    compute_sequence_properties,
    apply_mutation,
    apply_mutations,
    mutate_all_residues,
    save_sequences,
    AROMATIC_RESIDUES,
)

## 1. The Canonical FUS LCD Sequence

The FUS protein contains an N-terminal low-complexity domain (LCD, residues 1-214) that drives liquid-liquid phase separation. This region is enriched in:
- Tyrosine (Y): Primary "sticker" residues for aromatic interactions
- Glycine (G): Flexible spacers
- Serine (S): Polar residues
- Glutamine (Q): Polar residues

In [ ]:
# Display the FUS LCD sequence
print("FUS LCD Sequence (residues 1-214):")
print("=" * 60)

# Format with line numbers
for i in range(0, len(FUS_LCD_SEQUENCE), 60):
    chunk = FUS_LCD_SEQUENCE[i:i+60]
    print(f"{i+1:3d} {chunk}")

print(f"\nTotal length: {len(FUS_LCD_SEQUENCE)} residues")

In [ ]:
# Get the wild-type SequenceRecord
wt = get_variant("WT")
print(f"SequenceRecord: {wt}")
print(f"\nTyrosine positions (1-indexed): {wt.tyrosine_positions[:10]}... ({wt.tyrosine_count} total)")

## 2. Sequence Properties

In [ ]:
# Compute biophysical properties
props = compute_sequence_properties(wt.sequence)

print("FUS LCD Sequence Properties:")
print("=" * 40)
for key, value in props.items():
    if key != 'tyrosine_positions':  # Skip long list
        if isinstance(value, float):
            print(f"{key:25s}: {value:.3f}")
        else:
            print(f"{key:25s}: {value}")

In [ ]:
# Visualize amino acid composition
composition = wt.composition

# Sort by count
sorted_comp = dict(sorted(composition.items(), key=lambda x: -x[1]))

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#E74C3C' if aa in AROMATIC_RESIDUES else '#3498DB' for aa in sorted_comp.keys()]
ax.bar(sorted_comp.keys(), sorted_comp.values(), color=colors, edgecolor='black')
ax.set_xlabel('Amino Acid')
ax.set_ylabel('Count')
ax.set_title('FUS LCD Amino Acid Composition\n(Red = Aromatic)')
plt.tight_layout()
plt.show()

print(f"\nTop 5 residues: {list(sorted_comp.items())[:5]}")

In [ ]:
# Visualize tyrosine distribution
tyr_pos = props['tyrosine_positions']

fig, ax = plt.subplots(figsize=(14, 2))
ax.scatter(tyr_pos, [1] * len(tyr_pos), c='#E74C3C', s=100, marker='|', linewidths=2)
ax.set_xlim(0, len(wt.sequence))
ax.set_ylim(0.5, 1.5)
ax.set_yticks([])
ax.set_xlabel('Residue Position')
ax.set_title(f'Tyrosine Distribution in FUS LCD (n={len(tyr_pos)})')
plt.tight_layout()
plt.show()

print(f"Mean tyrosine spacing: {props['mean_tyrosine_spacing']:.1f} residues")

## 3. Mutation Utilities

We provide safe mutation functions that validate against the reference sequence.

In [ ]:
# Single mutation example
original_aa = wt.get_residue(144)
print(f"Residue at position 144: {original_aa}")

# Apply Y144S mutation
mutated_seq = apply_mutation(wt.sequence, "Y144S")
print(f"\nAfter Y144S mutation:")
print(f"Position 144: {mutated_seq[143]}")
print(f"Mutation verified: {mutated_seq[143] == 'S'}")

In [ ]:
# Multiple mutations
mutations = ["Y6S", "Y17S", "Y28S"]
multi_mutated = apply_mutations(wt.sequence, mutations)

print("Multiple Y→S mutations applied:")
for mut in mutations:
    pos = int(mut[1:-1])
    print(f"  Position {pos}: {wt.sequence[pos-1]} → {multi_mutated[pos-1]}")

In [ ]:
# Mutate all tyrosines
all_y_to_s, mutations_list = mutate_all_residues(wt.sequence, "Y", "S")

print(f"All Y→S mutations ({len(mutations_list)} total):")
print(f"First 5: {mutations_list[:5]}")
print(f"\nTyrosine count in mutated: {all_y_to_s.count('Y')}")
print(f"Serine count change: {wt.sequence.count('S')} → {all_y_to_s.count('S')}")

## 4. Variant Registry

Pre-defined variants for systematic analysis:
- **WT**: Wild-type FUS LCD
- **Y144S**: Single tyrosine-to-serine mutation
- **AllY_to_S**: All tyrosines mutated to serine (sticker disruption)
- **AllY_to_A**: All tyrosines mutated to alanine (aromatic knockout)
- **AllY_to_F**: All tyrosines mutated to phenylalanine (conservative control)

In [ ]:
# List all variants
print("Available variants:")
print("=" * 60)

for name in list_variants():
    variant = get_variant(name)
    print(f"\n{name}:")
    print(f"  Description: {variant.description}")
    print(f"  Tyrosines: {variant.tyrosine_count}")
    print(f"  Aromatics: {variant.aromatic_count}")
    if variant.mutations:
        print(f"  Mutations: {len(variant.mutations)} total")

In [ ]:
# Compare aromatic content across variants
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Tyrosine count
names = list(VARIANTS.keys())
tyr_counts = [VARIANTS[name].tyrosine_count for name in names]
axes[0].bar(names, tyr_counts, color='#F39C12', edgecolor='black')
axes[0].set_ylabel('Count')
axes[0].set_title('Tyrosine Count by Variant')
axes[0].tick_params(axis='x', rotation=45)

# Aromatic fraction
arom_frac = [VARIANTS[name].aromatic_fraction for name in names]
axes[1].bar(names, arom_frac, color='#9B59B6', edgecolor='black')
axes[1].set_ylabel('Fraction')
axes[1].set_title('Aromatic Fraction by Variant')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# FASTA output example
print("FASTA format for Y144S variant:")
print("=" * 60)
print(get_variant("Y144S").to_fasta())

## 5. Save Sequences

In [ ]:
# Save all sequences to data directory
output_dir = Path("../data/sequences")
save_sequences(output_dir, VARIANTS)

print(f"Sequences saved to {output_dir.absolute()}")
print("\nFiles created:")
for f in sorted(output_dir.glob("*")):
    print(f"  {f.name}")

## Summary

In this notebook we:
1. Loaded the canonical FUS LCD sequence (214 residues)
2. Analyzed sequence composition (27 tyrosines, ~12.6% aromatic)
3. Demonstrated safe mutation utilities
4. Built a variant registry with 5 sequences for systematic analysis
5. Saved sequences for downstream processing

**Next**: Notebook 02 - Force Field and Interaction Maps